# Explore the OpenSky API response

This notebook is for exploration only — poking at the raw API to decide cleaning rules. The actual pipeline logic lives in `ingestion.py`, `cleaning.py`, `database.py`, `main.py`, not here.

# Call the OpenSky API for our bounding box

In [ ]:
import requests
import pandas as pd

BBOX = {"lamin": 5.0, "lomin": 68.0, "lamax": 30.0, "lomax": 92.0}
resp = requests.get("https://opensky-network.org/api/states/all", params=BBOX, timeout=30)
resp.status_code

# Shape of the raw response

In [ ]:
payload = resp.json()
states = payload["states"] or []
len(states)

# Load into a DataFrame with named columns

In [ ]:
COLUMNS = [
    "icao24", "callsign", "origin_country", "time_position", "last_contact",
    "longitude", "latitude", "baro_altitude", "on_ground", "velocity",
    "true_track", "vertical_rate", "sensors", "geo_altitude", "squawk",
    "spi", "position_source",
]
df = pd.DataFrame(states, columns=COLUMNS)
df.head()

# How many rows and columns

In [ ]:
df.shape

# Data type of each column

In [ ]:
df.dtypes

# How many callsigns are missing or blank

In [ ]:
df['callsign'].isna().sum(), (df['callsign'].str.strip() == '').sum()

# How many rows have missing coordinates

In [ ]:
df[['latitude', 'longitude']].isna().sum()

# Range of altitude values (spot outliers before deciding clamp bounds)

In [ ]:
df['baro_altitude'].describe()

# Countries represented in this snapshot

In [ ]:
df['origin_country'].value_counts().head(15)

# On-ground vs airborne split

In [ ]:
df['on_ground'].value_counts()

# Check for exact duplicate (icao24, last_contact) pairs — confirms whether dedup logic in cleaning.py is actually needed

In [ ]:
df.duplicated(subset=['icao24', 'last_contact']).sum()